# 🦌 Doñana — 3. Test del detector

Evaluamos el mejor modelo de `02_train.ipynb` sobre el **test**, que no se ha usado ni para entrenar ni para elegir el modelo.

Métricas:
- **mAP@[.5:.95]** (COCO): media del AP con umbrales de IoU de 0.5 a 0.95; exige caja precisa **y** especie correcta.
- **mAP@0.5 / @0.75**: con un único umbral de IoU (0.5 = caja aproximada, 0.75 = caja ajustada).
- **mAP agnóstico**: todas las clases como "animal": ¿encuentra al animal, aunque se equivoque de especie?
- **Localización / acierto**: con la detección de mayor score de cada foto (cada foto tiene un único animal),
  % de fotos con la caja bien puesta (IoU ≥ 0.5) y % con caja y especie correctas.
- **AP por clase** y **mAP día / noche (IR)**.

> Las cajas de referencia son de MegaDetector (pseudo-etiquetas), no anotaciones manuales: el mAP mide cuánto se parece
> el modelo a MegaDetector + la etiqueta de especie. Aun así, la comparación entre modelos/aumentaciones es válida.

In [ ]:
!pip install -q -U "albumentations>=2.0.5" torchmetrics pycocotools

In [ ]:
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/donana")  # donana_data.py, donana_det.py y processed/ están aquí
    LOCAL_DIR = Path("/content")                         # disco local de Colab (rápido)
else:
    PROJECT_DIR = Path.cwd()
    LOCAL_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR))

PROCESSED_DIR = PROJECT_DIR / "processed"            # anotaciones, splits y caché comprimida
MODELS_DIR = PROJECT_DIR / "models"                  # checkpoints (en Drive, sobreviven a desconexiones)
IMAGES_DIR = LOCAL_DIR / "dona_24_1024"              # imágenes reducidas (lado mayor 1024 px)
CACHE_ZIP = PROCESSED_DIR / "dona_24_1024.zip"

import json
import numpy as np
import pandas as pd
import torch
import donana_data as dd
import donana_det as det

dd.ensure_image_cache(IMAGES_DIR, CACHE_ZIP)
annotations = pd.read_csv(PROCESSED_DIR / "annotations.csv")
CLASS_NAMES = json.loads((PROCESSED_DIR / "dataset_info.json").read_text())["class_names"]
NUM_CLASSES = len(CLASS_NAMES) + 1   # + fondo (clase 0)
train_df = annotations[annotations["split"] == "train"]
valid_df = annotations[annotations["split"] == "valid"]
test_df = annotations[annotations["split"] == "test"]
print(f"Clases ({NUM_CLASSES - 1}): {CLASS_NAMES}")
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")

## Parámetros

In [ ]:
model_path = MODELS_DIR / "faster_rcnn_v1_baseline" / "best.pt"
use_gpu = True
batch_size = 8
num_workers = 2
score_threshold = 0.5          # solo para los dibujos y la matriz de confusión (el mAP usa todas las detecciones)

## Cargar el modelo

In [ ]:
device = det.get_device(use_gpu)
model, ckpt = det.load_trained_model(model_path, device)
print({k: v for k, v in ckpt["config"].items() if k != "class_names"})
print(f"Mejor época: {ckpt['history']['best_epoch']} | mAP valid: {ckpt['history']['best_value']:.4f}")

## Resultados cuantitativos

In [ ]:
test_set = det.DonanaDataset(test_df, IMAGES_DIR)
test_loader = det.make_loader(test_set, batch_size, shuffle=False, num_workers=num_workers)

# Máscaras día / noche en el mismo orden que el dataset (una fila por imagen)
lighting = test_df.groupby("filePath", sort=False)["lighting"].first().values
subsets = {"día": lighting == "día", "noche": lighting == "noche (IR)"}

results = det.evaluate(model, test_loader, device, CLASS_NAMES, subsets=subsets)
pd.Series(det.summary_row(results)).round(4).to_frame("test")

In [ ]:
per_class = pd.DataFrame({
    "AP": results["ap_por_clase"],
    "imágenes test": test_df.groupby("label_name").size(),
}).loc[CLASS_NAMES]
per_class.round(3)

### Matriz de confusión por foto
Especie real vs especie de la detección principal. La última columna son los animales no detectados (o mal localizados).

In [ ]:
confusion = det.plot_confusion(results["_preds"], results["_gts"], CLASS_NAMES, score_threshold=0.3)

## Resultados cualitativos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision.ops import box_iou

def top_is_correct(pred, gt):
    if len(pred["boxes"]) == 0:
        return False
    top = int(pred["scores"].argmax())
    return (box_iou(pred["boxes"][top:top + 1], gt["boxes"]).max() >= 0.5).item() and int(pred["labels"][top]) == int(gt["labels"][0])

correct = np.array([top_is_correct(p, g) for p, g in zip(results["_preds"], results["_gts"])])
rng = np.random.default_rng(0)

def show_grid(indices, title):
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, indices):
        image, target = test_set[i]
        det.plot_predictions(image, results["_preds"][i], CLASS_NAMES, target=target,
                             score_threshold=score_threshold, ax=ax, title=f"{test_set.records[i]['filePath'].split('/')[-1]} | {lighting[i]}")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_grid(rng.choice(np.where(correct)[0], 8, replace=False), "Aciertos (verde = predicción, rojo = referencia)")
show_grid(rng.choice(np.where(~correct)[0], min(8, (~correct).sum()), replace=False), "Errores")

### Predecir una imagen cualquiera

In [ ]:
from PIL import Image
import torchvision.transforms.functional as TF

image_path = IMAGES_DIR / test_df.iloc[0]["filePath"]   # cambia por la ruta de cualquier foto
image = Image.open(image_path).convert("RGB")
with torch.no_grad():
    output = model([TF.to_tensor(image).to(device)])
det.plot_predictions(np.array(image), {k: v.cpu() for k, v in output[0].items()}, CLASS_NAMES, score_threshold=score_threshold)